# Outliers with the z-score method
Detecting outliers in `cgpa` with mean ± 3 standard deviations, then trimming and capping them.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import seaborn.objects as so
from sklearn.model_selection import train_test_split

# one look for every plot: Latin Modern font, white grid
THEME = {**sns.axes_style("whitegrid"), "font.family": "Latin Modern Roman", "font.size": 13}

## Load the data
1000 students: CGPA, marks in the placement exam (out of 100), and whether they were placed (1) or not (0).

In [ ]:
df = pd.read_csv("data/placement.csv")
print(df.shape)
df.sample(5, random_state=1)

## Check the shape of each column
The z-score method needs a roughly normal column.

In [ ]:
# put both columns into one long table: one row per (column, value)
long = df[["cgpa", "placement_exam_marks"]].melt(var_name="column", value_name="value")
(so.Plot(long, x="value")
   .facet(col="column").share(x=False, y=False)
   # histogram, each column with its own bins
   .add(so.Bars(alpha=0.5), so.Hist(stat="density", common_bins=False))
   # smooth density curve (KDE) on top
   .add(so.Line(), so.KDE(common_grid=False, common_norm=False))
   .layout(size=(10, 4)).theme(THEME))

In [ ]:
# skewness: near 0 = symmetric, above 0 = long right tail
df[["cgpa", "placement_exam_marks"]].skew()

`cgpa` is close to normal (skew about 0), `placement_exam_marks` is right-skewed. We use the z-score method on `cgpa` only.

## Approach 1: limits from the mean and standard deviation

In [ ]:
mean = df["cgpa"].mean()
std = df["cgpa"].std()
print("mean", round(mean, 2), " std", round(std, 2))
print("min ", df["cgpa"].min(), " max", df["cgpa"].max())

In [ ]:
# the limits: 3 standard deviations below and above the mean
upper_limit = mean + 3 * std
lower_limit = mean - 3 * std
print("upper limit", round(upper_limit, 4))
print("lower limit", round(lower_limit, 4))

In [ ]:
# rows outside the limits are outliers
is_outlier = (df["cgpa"] > upper_limit) | (df["cgpa"] < lower_limit)
df[is_outlier]

### Trimming

In [ ]:
# keep only the rows inside the limits (~ means "not")
trimmed = df[~is_outlier]
trimmed.shape

## Approach 2: the z-score column
Same limits, written as z-scores: $z = (x - \\text{mean}) / \\text{std}$, outlier if $z > 3$ or $z < -3$.

In [ ]:
df["cgpa_zscore"] = (df["cgpa"] - mean) / std
df.head()

In [ ]:
# z above 3: too high
df[df["cgpa_zscore"] > 3]

In [ ]:
# z below -3: too low
df[df["cgpa_zscore"] < -3]

In [ ]:
# trimming with the z-score: keep -3 <= z <= 3
trimmed_z = df[df["cgpa_zscore"].between(-3, 3)]
trimmed_z.shape

### Capping

In [ ]:
capped = df.copy()
# np.where(condition, value if true, value if false), nested twice:
# above the upper limit -> upper limit; below the lower limit -> lower limit; else unchanged
capped["cgpa"] = np.where(
    capped["cgpa"] > upper_limit,
    upper_limit,
    np.where(capped["cgpa"] < lower_limit, lower_limit, capped["cgpa"]),
)
print(capped.shape)
capped["cgpa"].describe()

In [ ]:
# the same thing in one line: clip pulls every value into [lower, upper]
same = df["cgpa"].clip(lower_limit, upper_limit)
(same == capped["cgpa"]).all()

## Better practice: learn the limits on the training set
The limits are learned from data, like a scaler's mean. We split first, learn them on the training rows, and apply the same limits to the test rows.

In [ ]:
train, test = train_test_split(df[["cgpa", "placement_exam_marks", "placed"]],
                               test_size=0.2, random_state=42)
# limits from the training set only
m, s = train["cgpa"].mean(), train["cgpa"].std()
lo, hi = m - 3 * s, m + 3 * s
print("training limits", round(lo, 2), round(hi, 2))

In [ ]:
# outliers in each part, judged with the training limits
for name, part in [("train", train), ("test", test)]:
    out = part[(part["cgpa"] < lo) | (part["cgpa"] > hi)]
    print(name, len(part), "rows,", len(out), "outliers:", out["cgpa"].tolist())

In [ ]:
# cap both parts with the training limits (trimming would only be done on train)
train_capped = train.assign(cgpa=train["cgpa"].clip(lo, hi))
test_capped = test.assign(cgpa=test["cgpa"].clip(lo, hi))
train_capped["cgpa"].agg(["min", "max"]), test_capped["cgpa"].agg(["min", "max"])